In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Task 1: Write your code here:
Q1_path = os.path.join(path, 'Q1_data.csv')

In [ ]:
Q1_data = pd.read_csv(Q1_path)

In [ ]:
# Task 2: Write your code here:.
Q1_data.head()

In [ ]:
# Task 3: Write your code here:
Q1_data.info()

In [ ]:
# Task 4: Write your code here:
Q1_data.describe()

In [ ]:
# Task 5: Write your code here:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(Q1_data, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
Q1_data = Q1_data.drop(columns="Order_ID", axis=1)

In [ ]:
Q1_data.columns


In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(Q1_data)

In [ ]:
Q1_data.info()

In [ ]:
catogrical_columns = ["Weather", "Traffic_Level", "Time_of_Day"]

Q1_data[catogrical_columns]

In [ ]:
for col in catogrical_columns:
    Q1_data[col] = Q1_data[col].fillna(Q1_data[col].mode()[0])

In [ ]:
num_columns = ["Courier_Experience_yrs", "Delivery_Time"]
for col in num_columns:
    Q1_data[col] = Q1_data[col].fillna(Q1_data[col].mean())

In [ ]:
missing_values = Q1_data.isnull().sum()
print("Columns with missing values:")
print(missing_values[missing_values > 0])

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(Q1_data)

In [ ]:
# 3. Do we have categorical columns?
def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))

label_encoders = encode_categorical_columns(Q1_data)

In [ ]:
from sklearn.preprocessing import LabelEncoder

all_catogrical_columns = ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]


label_encoders = {}
for col in all_catogrical_columns:
  le = LabelEncoder()
  Q1_data[col] = le.fit_transform(Q1_data[col])
  label_encoders[col] = le

Q1_data.info()

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = Q1_data.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
Q1_data[numerical_cols] = scaler.fit_transform(Q1_data[numerical_cols])
Q1_data.head()


In [ ]:
import seaborn as sns

In [ ]:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(Q1_data, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = Q1_data.drop("Delivery_Time", axis=1).astype(float)
y = Q1_data['Delivery_Time'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
X

In [ ]:
y

In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_index, test_index in kfold.split(X_train, y_train):
      # Split data into training and testing sets
  X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
  y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
  model.fit(X_Train, y_Train)
        # Predict on the test set
  y_pred = model.predict(X_Test)


    # Calculate metrics
  mae_scores.append(mean_absolute_error(y_Test, y_pred))
  rmse_scores.append(np.sqrt(mean_squared_error(y_Test, y_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
X.columns

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
                'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
y

In [ ]:
# Task 2: Write your code here:
# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(Q1_data['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: